# GitHub Marketplace ការប៉ាន់ស្មានម៉ូឌែល ជាមួយ Phi-4 Reasoning 

កំណត់ត្រានេះបង្ហាញរបៀបប្រើម៉ូឌែលពី GitHub Marketplace សម្រាប់ការប៉ាន់ស្មាន, ជាពិសេសដោយប្រើម៉ូឌែល Phi-4-reasoning របស់ Microsoft។

## ការតំឡើង និង ការកំណត់

ដំបូង យើងនឹងកំណត់បរិយាកាសរបស់យើង និងដំឡើងការពឹងផ្អែកដែលចាំបាច់។


In [ ]:
# Install required packages
!pip install requests python-dotenv
!pip install azure-ai-inference

### ការកំណត់ឯកសារ local.env

មុនពេលដំណើរការ notebook នេះ អ្នកត្រូវបង្កើតឯកសារ `local.env` នៅក្នុងថតដូចគ្នាជាមួយកំណត់ត្រានេះ ដែលមានអថេរដូចខាងក្រោម:

```
# GitHub Configuration
GITHUB_TOKEN=your_personal_access_token_here
GITHUB_INFERENCE_ENDPOINT=https://models.github.ai/inference
GITHUB_MODEL=microsoft/Phi-4-reasoning

# Azure OpenAI Configuration
AZURE_API_KEY=your_azure_api_key_here
AZURE_OPENAI_ENDPOINT=your_azure_endpoint_here
AZURE_OPENAI_MODEL=Phi-4-reasoning
```

**សេចក្តីណែនាំ:**

1. បង្កើតឯកសារថ្មីមានឈ្មោះ `local.env` នៅក្នុងថតដូចគ្នាជាមួយកំណត់ត្រានេះ
2. បន្ថែមអថេរបរិស្ថានចំនួនបីដែលបង្ហាញខាងលើ
3. ជំនួស `your_personal_access_token_here` ជាមួយ Token ការចូលផ្ទាល់ខ្លួន (Personal Access Token) របស់ GitHub
4. អ្នកអាចជាជម្រើសផ្លាស់ប្ដូរម៉ូឌែលទៅ `microsoft/Phi-4-mini-reasoning` សម្រាប់ម៉ូឌែលទំហំតូចជាង

**ចំណាំ:** Token របស់ GitHub ត្រូវការអាជ្ញាបណ្ណដែលសមរម្យ ដើម្បីចូលប្រើសេវាកម្មម៉ូឌែល AI


## បញ្ចូលអថេរបរិស្ថាន

យើងនឹងផ្ទុកអថេរបរិស្ថានពីឯកសារ `local.env` ដែលមាន GitHub token និងព័ត៌មានម៉ូឌែល។


In [ ]:
import os
import requests
import json
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

# Load variables from local.env file
load_dotenv('local.env')

# Access the environment variables - using values from local.env file
endpoint = os.getenv("GITHUB_INFERENCE_ENDPOINT")
model = os.getenv("GITHUB_MODEL")
token = os.getenv("GITHUB_TOKEN")
azuretoken = os.getenv("AZURE_KEY")
azureendpoint = os.getenv("AZURE_ENDPOINT")
azuremodel = os.getenv("AZURE_MODEL")
# Use fallback values if not found in local.env
if not endpoint:
    endpoint = "https://models.github.ai/inference"
    print("Warning: GITHUB_INFERENCE_ENDPOINT not found in local.env, using default value")

if not model:
    model = "microsoft/Phi-4-reasoning"
    print("Warning: GITHUB_MODEL not found in local.env, using default value")
    print("To change the model to Phi-4-mini-reasoning use \"microsoft/Phi-4-mini-reasoning\"")

if not token:
    raise ValueError("GITHUB_TOKEN not found in local.env file. Please add your GitHub token.")

print(f"Endpoint: {endpoint}")
print(f"Model: {model}")
print(f"azure_ai_image_generation_new.ipynb: {azureendpoint}")
print(f"azuremodel: {azuremodel}")
print(f"azuretoken available: {'Yes' if azuretoken else 'No'}")
print(f"Token available: {'Yes' if token else 'No'}")

## មុខងារជំនួយសម្រាប់ការប៉ាន់ស្មានម៉ូដែល

យើង​នឹង​បង្កើត​មុខងារ​ជំនួយ​ដើម្បីអន្តរកម្មជាមួយ GitHub inference API។


In [5]:
def generate_completion(prompt, model_id=model, temperature=0.7, max_tokens=10000):
    """Generate a completion using the GitHub inference API"""
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model_id,
        "prompt": prompt,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    try:
        # GitHub Models API requires a different endpoint structure
        api_url = f"{endpoint}/v1/chat/completions"
        print(f"Calling API at: {api_url}")
        
        # Modify payload for chat completions format
        chat_payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        
        response = requests.post(api_url, headers=headers, json=chat_payload)
        response.raise_for_status()  # Raise exception for 4XX/5XX errors
        result = response.json()
        
        if 'choices' in result and len(result['choices']) > 0:
            # Handle chat completions response format
            if 'message' in result['choices'][0] and 'content' in result['choices'][0]['message']:
                return result['choices'][0]['message']['content']
            # Fall back to the text field if available
            elif 'text' in result['choices'][0]:
                return result['choices'][0]['text']
            else:
                return f"Error: Could not extract content from response: {result['choices'][0]}"
        else:
            return f"Error: Unexpected response format: {result}"
    except Exception as e:
        print(f"Full error details: {str(e)}")
        return f"Error during API call: {str(e)}"

def format_conversation(messages):
    """Format a conversation for the model"""
    # For chat completion API, we'll just return the messages directly
    return messages

def generate_chat_completion(messages, model_id=model, temperature=0.7, max_tokens=1000):
    """Generate a completion using GitHub's chat completions API"""
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model_id,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    try:
        api_url = f"{endpoint}/v1/chat/completions"
        print(f"Calling API at: {api_url}")
        
        response = requests.post(api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        
        if 'choices' in result and len(result['choices']) > 0:
            if 'message' in result['choices'][0]:
                return result['choices'][0]['message']['content']
            else:
                return f"Error: Unexpected response format: {result['choices'][0]}"
        else:
            return f"Error: Unexpected response format: {result}"
    except Exception as e:
        print(f"Full error details: {str(e)}")
        return f"Error during API call: {str(e)}"
        
# For backward compatibility with existing code
def format_prompt_legacy(messages):
    """Format a conversation in text format (legacy method)"""
    formatted_prompt = ""
    
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        
        if role == "user":
            formatted_prompt += f"User: {content}\n\n"
        elif role == "assistant":
            formatted_prompt += f"Assistant: {content}\n\n"
        elif role == "system":
            formatted_prompt += f"{content}\n\n"
    
    formatted_prompt += "Assistant: "
    return formatted_prompt

## ឧទាហរណ៍ 1: តើត្រូវការផ្លែទំពាំងបាយជ្រុំប៉ុន្មានសម្រាប់អក្សរ r ចំនួន 9?

យើងនឹងរត់ឧទាហរណ៍សន្និដ្ឋានដំបូងរបស់យើង ដើម្បីសួរអំពីទំពាំងបាយជ្រុំ និងអក្សរ r។


In [6]:
example1_messages = [
    {"role": "system", "content": "You are a helpful AI assistant that answers questions accurately and concisely."},
    {"role": "user", "content": "How many strawberries do I need to collect 9 r's?"}
]

print("Messages:")
for msg in example1_messages:
    print(f"{msg['role']}: {msg['content']}")
print("\nGenerating response...\n")

# Use the new chat completion function directly with the messages
response1 = generate_chat_completion(example1_messages)
print("Response:")
print(response1)

Messages:
system: You are a helpful AI assistant that answers questions accurately and concisely.
user: How many strawberries do I need to collect 9 r's?

Generating response...

Calling API at: https://models.github.ai/inference/v1/chat/completions
Response:
<think>User says: "How many strawberries do I need to collect 9 r's?" This appears to be some riddle. Possibly the phrase "9 r's" might be a pun or reference to "r" letters. For example, "How many strawberries do I need to collect 9 r's?" It might be a pun on the phrase "strawberry, nine, r's" or "I need 9 r's" might be a riddle. Alternatively, perhaps the question "How many strawberries do I need to collect 9 r's?" might be a riddle where the answer is something like "9 strawberries" are needed if "strawberry" contains one "r" or two? Let's think: "Strawberry" letter count: S T R A W B E R R Y. Counting letter "r": "strawberry" has two "r"s, one in "straw" and one in "berry" but wait, let's check: "strawberry" letters: S, T, R, A

### វិភាគឧទាហរណ៍ 1

នៅក្នុងឧទាហរណ៍នេះ ម៉ូឌែលត្រូវយល់ថា ពាក្យ "strawberry" មានអក្សរ 'r' ពីរ។ ដូច្នេះ ដើម្បីប្រមួលអក្សរ 'r' ចំនួន 9 អ្នកនឹងត្រូវការ 5 strawberries (ដែលមានអក្សរ 'r' សរុប 10), ឬ 4.5 strawberries ដើម្បីទទួលបានអក្សរ 'r' ត្រឹមត្រូវ 9។

មកមើលថាតើម៉ូឌែល Phi-4-reasoning ដោះស្រាយបញ្ហានេះយ៉ាងដូចម្តេច។


## ឧទាហរណ៍ 2: ដោះស្រាយសំនួរអាថ៌កំបាំង

ឥឡូវនេះ យើងសាកល្បងឧទាហរណ៍ស្មុគស្មាញជាងនេះ - សំនួរអាថ៌កំបាំងសម្រាប់ការទទួលស្គាល់លំនាំដែលមានឧទាហរណ៍ច្រើន។


In [7]:
example2_messages = [
    {"role": "system", "content": "You are a helpful AI assistant that solves riddles and finds patterns in sequences."},
    {"role": "user", "content": "I will give you a riddle to solve with a few examples, and something to complete at the end"},
    {"role": "user", "content": "nuno Δημήτρης evif Issis 4"},
    {"role": "user", "content": "ntres Inez neves Margot 4"},
    {"role": "user", "content": "ndrei Jordan evlewt Μαρία 9"},
    {"role": "user", "content": "nπέντε Kang-Yuk xis-ytnewt Nubia 21"},
    {"role": "user", "content": "nπέντε Κώστας eerht-ytnewt Μανώλης 18"}, 
    {"role": "user", "content": "nminus one-point-two Satya eno Bill X."},
    {"role": "user", "content": "What is a likely completion for X that is consistent with examples above?"}
]

print("Messages:")
for msg in example2_messages:
    print(f"{msg['role']}: {msg['content'][:50]}...")
print("\nGenerating response...\n")

response2 = generate_chat_completion(example2_messages, temperature=0.2, max_tokens=10000)
print("Response:")
print(response2)

Messages:
system: You are a helpful AI assistant that solves riddles...
user: I will give you a riddle to solve with a few examp...
user: nuno Δημήτρης evif Issis 4...
user: ntres Inez neves Margot 4...
user: ndrei Jordan evlewt Μαρία 9...
user: nπέντε Kang-Yuk xis-ytnewt Nubia 21...
user: nπέντε Κώστας eerht-ytnewt Μανώλης 18...
user: nminus one-point-two Satya eno Bill X....
user: What is a likely completion for X that is consiste...

Generating response...

Calling API at: https://models.github.ai/inference/v1/chat/completions
Response:
<think>We are given a riddle with examples. The riddle is: "I will give you a riddle to solve with a few examples, and something to complete at the end". The examples are:

1. "nuno Δημήτρης evif Issis 4"
2. "ntres Inez neves Margot 4"
3. "ndrei Jordan evlewt Μαρία 9"
4. "nπέντε Kang-Yuk xis-ytnewt Nubia 21"
5. "nπέντε Κώστας eerht-ytnewt Μανώλης 18"
6. "nminus one-point-two Satya eno Bill X."

We are asked: "What is a likely completion for X that is

### ការវិភាគនៃឧទាហរណ៍ទី 2

កំហ៊ូនេះទាមទារឱ្យស្គាល់លំនាំស្មុគស្មាញកាលពីភាសាច្រើននិងការបង្ហាញលេខក្នុងទ្រង់ទ្រាយផ្សេងៗ។ យើងមកផ្តាច់ចែកអំពីអ្វីដែលម៉ាស៊ីនមូលដ្ឋានត្រូវយល់៖

1. សម្គាល់អក្សរត្រឡប់ក្រោយក្នុងពាក្យដូចជា "evif" (ប្រាំ)
2. ទទួលស្គាល់លេខក្នុងភាសាផ្សេងៗ (ឧ., "uno" ក្នុងភាសាអេស្បាញ, "πέντε" ក្នុងភាសាក្រិក)
3. ស្វែងរកទំនាក់ទំនងរវាងលេខនានា និងខ្ទង់ចុងក្រោយ

លំនាំនេះបង្ហាញថាមានប្រតិបត្តិការគណិតវិទ្យាដែលធ្វើលើតំលៃដែលបានតំណាងក្នុងភាសា និងទ្រង់ទ្រាយផ្សេងៗ។


## ការសាកល្បងជាមួយប៉ារ៉ាម៉ែត្រ ផ្សេងៗ

មកសាកល្បងឧទាហរណ៍ទីពីរម្ដងទៀតជាមួយការកំណត់សីតុណ្ហភាពផ្សេងៗ ដើម្បីមើលថាវាធ្វើឲ្យការឆ្លើយតបរបស់ម៉ូដែលផ្លាស់ប្តូរយ៉ាងដូចម្តេច។


In [ ]:
# Try with a higher temperature for more creative responses
response_creative = generate_chat_completion(example2_messages, temperature=0.9, max_tokens=20000)
print("Response with higher temperature (0.9):")
print(response_creative)

## បង្កើតឧទាហរណ៍របស់អ្នកឯង

អ្នកអាចបង្កើតឧទាហរណ៍របស់អ្នកឯង ដើម្បីសាកល្បងសមត្ថភាពក្នុងការពិចារណារបស់ម៉ូដែល។ សូមព្យាយាមកែប្រែពាក្យបញ្ចូល ឬបង្កើតស្ថានការណ៍ថ្មីទាំងស្រុងខាងក្រោម។


In [ ]:
# Define your custom prompt here
custom_messages = [
    {"role": "system", "content": "You are a helpful AI assistant that can solve complex problems."},
    {"role": "user", "content": "Your custom prompt here"}
]

# Uncomment the lines below to run your custom prompt
# custom_response = generate_chat_completion(custom_messages)
# print("Response to custom prompt:")
# print(custom_response)

## សន្និដ្ឋាន

កំណត់ត្រានេះបានបង្ហាញពីវិធីប្រើម៉ូឌែលនៅលើ GitHub Marketplace សម្រាប់ការសន្មត, ជាក់លាក់ការប្រើម៉ូឌែល Phi-4-reasoning ដើម្បីដោះស្រាយបញ្ហាផ្លូវហេតុ និងសំណួរអាថ៌កំបាំង។

ចំណុចសំខាន់ៗដែលទទួលបាន:
1. ការរៀបចំការផ្ទៀងផ្ទាត់អត្តសញ្ញាណជាមួយសញ្ញា GitHub
2. រៀបចំទ្រង់ទ្រាយនៃសំណើ (prompts) សម្រាប់ការសន្មតដែលមានប្រសិទ្ធភាពបំផុត
3. គ្រប់គ្រងប៉ារ៉ាម៉ែត្រម៉ូឌែល ដូចជា temperature ដើម្បីគ្រប់គ្រងការផ្លាស់ប្តូរនៃចម្លើយ
4. សាកល្បងសមត្ថភាព reasoning របស់ម៉ូឌែលលើបញ្ហាប្រភេទផ្សេងៗ

ចងចាំរក្សា GitHub token របស់អ្នកឲ្យមានសុវត្ថិភាព និងកុំចែករំលែកវានៅក្នុងឃ្លាំងកូដសាធារណៈ ឬសៀវភៅកំណត់ត្រា។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារ​នេះ​ត្រូវបាន​ប្រែសម្រួល​ដោយ​ប្រើ​សេវាកម្ម​ប្រែសម្រួល AI [Co-op Translator](https://github.com/Azure/co-op-translator). ទោះ​យើង​ព្យាយាម​ឲ្យ​មាន​ភាព​ត្រឹមត្រូវ សូម​ជ្រាបថា ការប្រែសម្រួល​អូតូម៉ាទិច​អាច​មានកំហុស ឬ​មិន​ត្រឹមត្រូវ។ ឯកសារដើម​ក្នុង​ភាសាដើម​គួរត្រូវបាន​គេ​យក​ជា​ប្រភព​ផ្លូវការ។ សម្រាប់ព័ត៌មានសំខាន់ យើងសូមណែនាំឲ្យប្រើការបកប្រែដោយមនុស្សវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយ ដែលកើតឡើងពីការប្រើប្រាស់ការប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
